# Auto-differentiation

Consider the following function, which is a (crude) approximation to the equation of state for a degenerate gas + ideal gas nuclei (we used this in {doc}`ODEs/application-xrb-one-zone`:

In [1]:
import numpy as np

In [16]:
k_B = 1.38e-16  # erg / K
h = 6.67e-27  # erg s
m_e = 9.11e-28  # g
m_u = 1.66e-24  # g

# electron degeneracy
K = h**2 /(20 * m_e) * (3.0 / np.pi)**(2./3.) * m_u**(-5./3.)
    
# we'll use pure He for the compoosition
mu_I = 4.0
mu_e = 2.0

In [25]:
def p(rho, T):

    # ions are ideal gas
    p_ion = rho * k_B * T / (mu_I * m_u)

    # electrons can be ideal gas or degenerate -- we use an approximation to blend
    # the two regimes
    p_e_ideal = rho * k_B * T / (mu_e * m_u)
    p_e_deg = K * (rho / mu_e)**(5./3.)
    p_e = np.sqrt(p_e_ideal**2 + p_e_deg**2)

    return p_ion + p_e

Given a $\rho, T$, we can get the total pressure.  But what if we wanted $\partial p/\partial \rho |_T$ or
$\partial p/\partial T|_\rho$?

We could use a finite-difference, like we did in {doc}`basics/diff-int/differentiation`.  But for complex functions,this can be slow, and suffer from accumulation of round-off error.

In [26]:
rho = 1.e5
T = 1.e7

pres = p(rho, T)

eps = 1.e-8
drho = rho * eps

dpdrho = (p(rho+drho, T) - p(rho-drho, T)) / (2.0 * drho)

(pres, dpdrho)

(np.float64(7.124649467208393e+20), np.float64(1.1719208861696e+16))

## Introduction to automatic differentiation

The basic idea of [automatic differentiation](https://en.wikipedia.org/wiki/Automatic_differentiation) or "autodiff"

```{note}
We will implement "forward mode" autodiff, which works well when we are only differentiating with respect to a few variables.
```

We use [dual numbers](https://en.wikipedia.org/wiki/Automatic_differentiation#Automatic_differentiation_using_dual_numbers) to propagate both the function value and its derivative at once.  This essentially just uses the chain rule automatically for us.

For a quantity $x$, we replace it with $x + x'\epsilon$, where $\epsilon \ll 1$ such that $\epsilon^2 \sim 0$.  Then we have:

* addition: $(x + x'\epsilon) + (y + y'\epsilon) = x + y + (x' + y')\epsilon$
* subtraction: $(x + x'\epsilon) - (y + y'\epsilon) = x - y + (x' - y')\epsilon$
* multiplication: $(x + x'\epsilon) \cdot (y + y'\epsilon) = xy + (xy' + yx')\epsilon + \mathcal{O}(\epsilon^2)$
* division:

  $$\frac{x + x'\epsilon}{y + y'\epsilon} = \frac{\frac{x}{y} + \frac{x'\epsilon}{y}}{1 + \frac{y'\epsilon}{y}} \approx \left (\frac{x}{y} + \frac{x'\epsilon}{y} \right ) \left (1 - \frac{y'\epsilon}{y} \right ) = \frac{x}{y} + \left (\frac{x'y - xy'}{y^2} \right )\epsilon$$

We see that these just give the expression from calculus that we are familiar with.

What about complex functions, like $x^y$?

We can use the same ideas.

Start with:

$$q = (x + x'\epsilon)^{(y + y'\epsilon)}$$

It's easiest to take the log first:

$$
\log q &= (y + y'\epsilon) \log(x + x'\epsilon) \\
       &= (y + y'\epsilon) \log \left [ x \left (1 + \frac{x'\epsilon}{x} \right ) \right ] \\
       &= (y + y'\epsilon) \left [\log x + \underbrace{\log \left ( 1 + \frac{x'\epsilon}{x} \right )}_{\sim x'\epsilon/x} \right ] \\
       &= y\log x + y'\epsilon \log x + \frac{yx'}{x} \epsilon + \mathcal{O}(\epsilon^2) \\
       &= \log x^y + \left (\log x^{y'} + \frac{yx'}{x} \right )\epsilon
$$

Now we recover $q = e^{\log q}$:

$$q &= \exp\left \{ \log x^y + \left (\log x^{y'} + \frac{yx'}{x} \right )\epsilon\right \} \\
    &= x^y \cdot \exp \left \{\left (\log x^{y'} + \frac{yx'}{x} \right )\epsilon\right \}
$$

Now using $e^x \sim 1 + x$ for $|x| \ll 1$:

$$q = x^y \left [1 + \left (y^\prime \log x + \frac{yx'}{x} \right )\epsilon \right ]$$

we would implement this in the `__pow__` method of our `Dual` class.

## Implementation

We can implement most of the dual arithmetic by overloading operators.  For some special functions,
like `sqrt()`, we will have to write our own function.

In [33]:
class Dual:
    """Carry a quantity that can propagate its derivatve.

    We carry a scalar derivative, so that means we are always
    differentiating with respect to the same quantity.

    """

    def __init__(self, x, deriv=1):
        self.value = x
        self.deriv = deriv

    def __abs__(self):
        if self.value > 0:
            return Dual(self.value, self.deriv)
        if self.value < 0:
            return Dual(-self.value, -self.deriv)
        return Dual(abs(self.value), 0.0)

    def __lt__(self, other):
        if isinstance(other, Dual):
            return self.value < other.value
        return self.value < other

    def __gt__(self, other):
        if isinstance(other, Dual):
            return self.value > other.value
        return self.value > other

    def __eq__(self, other):
        if isinstance(other, Dual):
            return self.value == other.value
        return self.value == other

    def __add__(self, other):
        if isinstance(other, Dual):
            return Dual(self.value + other.value, self.deriv + other.deriv)
        return Dual(self.value + other, self.deriv)

    def __radd__(self, other):
        return self.__add__(other)

    def __sub__(self, other):
        if isinstance(other, Dual):
            return Dual(self.value - other.value, self.deriv - other.deriv)
        return Dual(self.value - other, self.deriv)

    def __rsub__(self, other):
        if isinstance(other, Dual):
            return other.__sub__(self)
        return Dual(other - self.value, -self.deriv)

    def __neg__(self):
        return Dual(-self.value, -self.deriv)

    def __mul__(self, other):
        if isinstance(other, Dual):
            # Product rule, using the original operand values.
            return Dual(self.value * other.value,
                        self.value * other.deriv + other.value * self.deriv)
        return Dual(self.value * other, self.deriv * other)

    def __rmul__(self, other):
        return self.__mul__(other)

    def __truediv__(self, other):
        if isinstance(other, Dual):
            # Quotient rule, using the original operand values.
            return Dual(self.value / other.value,
                        -self.value / other.value**2 * other.deriv + self.deriv / other.value)
        return Dual(self.value / other, self.deriv / other)

    def __rtruediv__(self, other):
        if isinstance(other, Dual):
            return other.__truediv__(self)
        return Dual(other / self.value, -other / self.value**2 * self.deriv)

    def __pow__(self, y):
        if isinstance(y, Dual):
            if self.value <= 0:
                raise ValueError("Dual exponents require a positive base")
            value = self.value**y.value
            deriv = value * (y.deriv * np.log(self.value)
                             + y.value * self.deriv / self.value)
            return Dual(value, deriv)

        if y == 0:
            return Dual(1.0, 0.0)
        return Dual(self.value**y,
                    y * self.value**(y - 1.0) * self.deriv)


def sqrt(x):
    """Compute sqrt(x)."""
    if isinstance(x, Dual):
        value = np.sqrt(x.value)
        return Dual(value, 0.5 * x.deriv / value)
    return np.sqrt(x)

```{important}
We need to be sure to call our version of `sqrt()` instead of the numpy version, so the `Dual`
will properly be used.
```

Here's our same equation of state as above, but we replace `np.sqrt()` with our new `sqrt()`.

In [28]:
def p(rho, T):

    # ions are ideal gas
    p_ion = rho * k_B * T / (mu_I * m_u)

    # electrons can be ideal gas or degenerate -- we use an approximation to blend
    # the two regimes
    p_e_ideal = rho * k_B * T / (mu_e * m_u)
    p_e_deg = K * (rho / mu_e)**(5./3.)
    p_e = sqrt(p_e_ideal**2 + p_e_deg**2)

    return p_ion + p_e

In [29]:
rho = Dual(1.e5)
T = 1.e7

In [30]:
p = p(rho, T)

In [31]:
p.value

np.float64(7.124649467208393e+20)

In [32]:
p.deriv

np.float64(1.1719208872136024e+16)